# Data ETL and exploration/preprocessing

## Loading and Exploring Data

### Guidelines

1. Load all the kcmetro text data independently and drop irrelevant information
2. Merge all data 
3. Explore data and calendar/caldates/service id in particular
4. Create function to calculate frequencies from schedule data
5. create function to map routes that match frequency target 

Libraries to Use:

-numpy and pandas 
-geopandas and folium
-datetime 
-os and shutils 
-arcpy

In [261]:
###installs

%pip install folium pandas numpy geopandas requests

Note: you may need to restart the kernel to use updated packages.


In [262]:
#imports
import numpy as np, pandas as pd, requests, json, os, zipfile
from pathlib import Path
import folium as fl, geopandas as gp 
from datetime import datetime



In [263]:
gtfs_url = r'https://gtfs.sound.obaweb.org/prod/gtfs_puget_sound_consolidated.zip'
output_dir = '../data/pgc_gtfs'

def download_and_unzip(url, out_dir):
    zip_dir = output_dir + '/gtfs_consolidated.zip'
    response = requests.get(url)
    with open (zip_dir,'wb') as f:
        f.write(response.content)

    with zipfile.ZipFile(zip_dir, 'r') as zip_ref:
        zip_ref.extractall(output_dir)

download_and_unzip(gtfs_url, output_dir)

In [264]:
###loading data
trips = pd.read_csv(r'../data/pgc_gtfs/trips.txt')
stop_times = pd.read_csv(r'../data/pgc_gtfs/stop_times.txt', dtype={'trip_id': str, 'stop_id': str, 'stop_headsign': str})
stops = pd.read_csv(r'../data/pgc_gtfs/stops.txt')
routes = pd.read_csv(r'../data/pgc_gtfs/routes.txt')
cal = pd.read_csv(r'../data/pgc_gtfs/calendar.txt')
cal_dates = pd.read_csv(r'../data/pgc_gtfs/calendar_dates.txt')

In [265]:
###dropping unnecessary information

trips_clean = trips[['route_id','service_id','trip_id','trip_headsign', 'direction_id','block_id','shape_id']]
stop_times_clean = stop_times[['trip_id','arrival_time','departure_time','stop_id','stop_sequence','shape_dist_traveled','timepoint']]
stops_clean = stops[['stop_id','stop_code','stop_name','stop_lat','stop_lon']]
routes_clean = routes[['route_id','agency_id','route_short_name','route_desc','route_type','route_color','route_text_color']]

###cal and calendar dates are to remain unchanged until I explore the data more thoroughly 
###set the variables to the clean dataframes

trips = trips_clean
stop_times = stop_times_clean
stops = stops_clean
routes = routes_clean


###find and set types
stop_times.arrival_time = pd.to_timedelta(stop_times.arrival_time)
stop_times.departure_time = pd.to_timedelta(stop_times.departure_time)



In [266]:
###merge together data

#merge all files one by one 

data_merged = pd.merge(trips, stop_times, how='left', on='trip_id')
data_merged = pd.merge(data_merged, stops, how='left', on='stop_id')
data_merged = pd.merge(data_merged, routes, how='left', on='route_id')

###filter to only kcm bus routes 
data_merged = data_merged.loc[data_merged.agency_id.isin([1,23,40])]

data_merged.agency_id.unique()

array([ 1, 23, 40])

### Data Exploration

In [267]:
###let us explore the service id to see which routes have which ids

route_servs = pd.DataFrame(data_merged.groupby('route_short_name').service_id.unique())
route_servs['unique_service_ids'] = route_servs.service_id.apply(lambda x: x.shape[0])

###put unique service ids in the front of the dataframe 
route_servs = route_servs.iloc[:,[1,0]]

data_merged.service_id.value_counts()

service_id
130540                                                  425771
25266                                                   331109
25530                                                   311471
2295                                                     20862
128245                                                   20474
                                                         ...  
SNDR_2026-10-04_20260813_Sounder_Seahawks1325_Sunday        27
SNDR_2026-11-08_20260813_Sounder_Seahawks1325_Sunday        27
SNDR_2026-12-13_20260813_Sounder_Seahawks1325_Sunday        27
87923                                                       25
108258                                                      10
Name: count, Length: 84, dtype: int64

As we can see, the most popular service ids are 130540, 25266, and 25530 - corresponding to weekday,sat, and sun service

In [268]:
####find calendar dates to confirm my intuition 
cal_serv = cal

###add total weekday count and set to one for boolean comparisons 
cal_serv['weekday'] = cal_serv.drop(columns=['service_id', 'start_date','end_date', 'saturday','sunday']).apply(lambda x: any(x), axis=1)

####drop individual day of the workweek columns 
cal_serv = cal_serv.drop(columns=cal_serv.columns[1:6])

### us DF.dot to multiply 
cal_serv['dow'] = cal_serv[['saturday','sunday','weekday']].dot(cal_serv.columns[[1,2,5]] + '_').str.lower().apply(lambda x: x.strip('_'))

###find services that are weekday, weekend, or special service
cal_serv = cal_serv.drop(columns=['saturday','sunday','weekday'])
cal_serv.dow = cal_serv.dow.replace('', 'special')

###make service id an str
cal_serv.service_id = cal_serv.service_id.astype(str)

###save service ids by dow for further exploration
cal_serv.to_csv('../data/serv_dow.csv')

###display 
cal_serv.dow.unique()


<ArrowStringArray>
['special', 'weekday', 'sunday', 'saturday', 'sunday_weekday',
 'saturday_sunday']
Length: 6, dtype: str

### Frequency Calculations/Functions and Verification

We will explore the data by using various methods to try to calculate frequency and then we will validate. 

In [269]:
##lets use groupby in pandas to filter/sort the data we desire and limit it to a single stop + dir + service_id 
data_merged = data_merged.sort_values(by=['stop_id', 'arrival_time'])

####group by stop id and sort by time of arrival 
data_merged['prev_depart'] = data_merged.groupby(['agency_id', 'route_id', 'service_id', 'direction_id', 'stop_id']).departure_time.shift(1)
data_merged['gap'] = data_merged['departure_time'] - data_merged['prev_depart']
route_freq = pd.DataFrame(data_merged.groupby('route_short_name').gap.mean().sort_values(ascending=True))

##group by stop sequence and sort by time of arrival
data_merged['prev_depart'] = data_merged.groupby(['route_id', 'service_id', 'direction_id', 'stop_sequence']).departure_time.shift(1)
data_merged['gap'] = data_merged['departure_time'] - data_merged['prev_depart']
route_freq['by_sequence'] = pd.DataFrame(data_merged.groupby('route_short_name').gap.mean().sort_values(ascending=True))


### we will now find the differences between sorting by stop sequence and sorting by stop_id 
route_freq['dif'] = np.abs(route_freq['by_sequence'] - route_freq['gap'])

###display the differences based on calculation type. 
###it would seem sensible to me that stop id would be the better variable to use as lines can start at various stop ids first (changing stop seq no.)
route_freq.groupby('route_short_name').gap.mean().sort_values()

route_short_name
Shuttle   0 days 00:08:00.216606
1 Line    0 days 00:10:04.138358
G Line    0 days 00:10:08.729701
2 Line    0 days 00:10:11.311608
E Line    0 days 00:12:15.598914
                   ...          
982                          NaT
986                          NaT
987                          NaT
988                          NaT
989                          NaT
Name: gap, Length: 163, dtype: timedelta64[us]

From here we will validate data using a few routes and accurate schedule data

In [270]:
###find data from rapidrides to compare


rr_data = data_merged.loc[data_merged.route_short_name.str.match(r'\w Line')]
rr_freq = route_freq.loc[route_freq.index.str.match(r'\w Line')]


In [271]:
###function for calculating frequencies based on a time range and a label (departure_time or arrival_time)
def freq_calc(df, label='departure_time', start_hr=0, end_hr=30):

    ### only locate columns we need for the calculations to save memory and increase run speed
    df_lite = df[['route_id','trip_id','stop_id','route_short_name', 'service_id','direction_id', label]]

    start_time = pd.Timedelta(hours=start_hr)
    end_time = pd.Timedelta(hours=end_hr)
    ###convert departure_time to timedelta
    df_lite.departure_time = pd.to_timedelta(df_lite.departure_time)
    ###restrict to time range
    df_lite = df_lite.loc[(df_lite.departure_time >= start_time) & (df_lite.departure_time <= end_time)]

    ####group by stop id and sort by time of arrival 
    df_lite['prev_depart'] = df_lite.groupby(['route_id', 'service_id', 'direction_id', 'stop_id']).departure_time.shift(1)
    df_lite['gap'] = df_lite['departure_time'] - df_lite['prev_depart']

    ###calculate frequencies 
    route_freq = pd.DataFrame(df_lite.groupby('route_short_name').gap.mean())

    route_freq['mean_gap'] = route_freq['gap']
    route_freq = route_freq.drop(columns='gap')

    route_freq['median_gap'] = df_lite.groupby('route_short_name').gap.median()

    return route_freq


    

In [272]:
###function for calculating first and last arrivals 

def first_last(df, label='departure_time', start_hr=0, end_hr=30):

    ### only locate columns we need for the calculations to save memory and increase run speed
    df_lite = df[['route_id','trip_id','stop_id','route_short_name', 'service_id','direction_id', label]]

    start_time = pd.Timedelta(hours=start_hr)
    end_time = pd.Timedelta(hours=end_hr)
    ###convert departure_time to timedelta
    df_lite.departure_time = pd.to_timedelta(df_lite.departure_time)
    ###restrict to time range
    df_lite = df_lite.loc[(df_lite.departure_time >= start_time) & (df_lite.departure_time <= end_time)]

    ####find first and last departure times for each route
    fl_df = df_lite.groupby('route_short_name')[label].agg(['min', 'max']).rename(columns={'min': 'first_departure', 'max': 'last_departure'})

    return fl_df

In [273]:
###calculate frequencies for each day of the week and time range and save to csv files for later use in mapping/streamlit dashboard
data_merged

for dow in ['weekday','saturday','sunday']:
    for times in [(0,30),(6,9),(9,15),(15,19),(19,24),(20,30)]:
        start_hr = times[0]
        end_hr = times[1]
        print(f'Calculating frequencies for {dow} from {start_hr} to {end_hr}')

        serv_ids = cal_serv.loc[cal_serv.dow == dow].service_id.unique()
        data_dow = data_merged.loc[data_merged.service_id.isin(serv_ids)]

        freq_df = freq_calc(data_dow, label='departure_time', start_hr=start_hr, end_hr=end_hr)
        freq_df[['first','last']] = first_last(data_dow, label='departure_time', start_hr=start_hr, end_hr=end_hr)
        freq_df['trip_count'] = data_dow.groupby('route_short_name').trip_id.nunique()

        freq_df.to_csv(f'../data/freq_data/freq_{dow}_{start_hr}_{end_hr}.csv')




Calculating frequencies for weekday from 0 to 30
Calculating frequencies for weekday from 6 to 9
Calculating frequencies for weekday from 9 to 15
Calculating frequencies for weekday from 15 to 19
Calculating frequencies for weekday from 19 to 24
Calculating frequencies for weekday from 20 to 30
Calculating frequencies for saturday from 0 to 30
Calculating frequencies for saturday from 6 to 9
Calculating frequencies for saturday from 9 to 15
Calculating frequencies for saturday from 15 to 19
Calculating frequencies for saturday from 19 to 24
Calculating frequencies for saturday from 20 to 30
Calculating frequencies for sunday from 0 to 30
Calculating frequencies for sunday from 6 to 9
Calculating frequencies for sunday from 9 to 15
Calculating frequencies for sunday from 15 to 19
Calculating frequencies for sunday from 19 to 24
Calculating frequencies for sunday from 20 to 30


## Scripts: UNDER CONSTRUCTION

In [ ]:
###getting initial data to compare
"""
The data pipeline will be as follows: 
    1. Download and unzip GTFS data
    2. use agency_id to filter only to certain agencies - i.e. city of seattle, kc metro, etc
    3. Merge all the data into one dataframe - 
    4. calc dow by calendar.txt
    5. Filter data by time range and dow
    6. calculate frequencies for each route 
"""

"""
AGENCY_FILTERS: kcmetro, sound transit, city of seattle
"""
AGENCY_FILTERS = [1,40,23]
ALL_HOURS = [0, 30]
ALL_DOW = ['weekday','saturday','sunday']
WEEKDAY = ['weekday']
SATURDAY = ['saturday']
SUNDAY = ['sunday']

import pandas as pd, urllib.request, zipfile, os, requests
from pathlib import Path
from datetime import datetime as dt

def download_and_unzip(url, out_dir):

    ###make dir so that it exists 
    Path(out_dir).mkdir(exist_ok=True)

    zip_dir = output_dir + '/gtfs_consolidated.zip'
    response = requests.get(url)
    with open (zip_dir,'wb') as f:
        f.write(response.content)

    with zipfile.ZipFile(zip_dir, 'r') as zip_ref:
        zip_ref.extractall(output_dir)

def build_feed(filters=AGENCY_FILTERS, hours=ALL_HOURS):
    """
    This function will build a feed of data for the specified agency ids.
    It will download the GTFS data, extract it, and load it into a pandas dataframe.
    It will then filter the data to only include the specified agency ids.
    """
    gtfs_url = r'https://gtfs.sound.obaweb.org/prod/gtfs_puget_sound_consolidated.zip'
    output_dir = '../data/pgc_gtfs'

    download_and_unzip(gtfs_url, output_dir)
    agencies = pd.read_csv(r'../data/pgc_gtfs/agency.txt').query('agency_id in @filters')
    routes = pd.read_csv(r'../data/pgc_gtfs/routes.txt').query('agency_id in @agencies.agency_id')
    trips = pd.read_csv(r'../data/pgc_gtfs/trips.txt').query('route_id in @routes.route_id')
    stop_times = pd.read_csv(r'../data/pgc_gtfs/stop_times.txt', dtype={'trip_id': str, 'stop_id': str, 'stop_headsign': str}).query('trip_id in @trips.trip_id')
    stops = pd.read_csv(r'../data/pgc_gtfs/stops.txt').query('stop_id in @stop_times.stop_id')

    agencies_clean = agencies[['agency_id', 'agency_name', 'agency_url']]
    routes_clean = routes[['route_id','agency_id','route_short_name','route_desc','route_type','route_color','route_text_color']]
    trips_clean = trips[['route_id','service_id','trip_id','trip_headsign', 'direction_id','block_id','shape_id']]
    stop_times_clean = stop_times[['trip_id','arrival_time','departure_time','stop_id','stop_sequence','shape_dist_traveled','timepoint']]
    stops_clean = stops[['stop_id','stop_code','stop_name','stop_lat','stop_lon']]
   

    """
    returns a dictionary of dataframes for the specified agency ids with only necesssary data 
    """
    return {
        'agencies': agencies_clean,
        'routes' : routes_clean,
        'trips': trips_clean,
        'stop_times': stop_times_clean,
        'stops': stops_clean
    }

def calendar_parse(dir):
    """
    This function will load the calendar.txt file and parse it to find the dow of the service_id 's 
    """
    file = os.path.join(dir, 'calendar.txt')
    cal = pd.read_csv(file)

    ###add total weekday count and set to one for boolean comparisons 
    cal['weekday'] = cal.drop(columns=['service_id', 'start_date','end_date', 'saturday','sunday']).apply(lambda x: any(x), axis=1)
    ####drop individual day of the workweek columns 
    cal = cal.drop(columns=cal.columns[1:6])

    ### us DF.dot to multiply weekday by sat by sun to get the days of week the service runs on 
    cal['dow'] = cal[['saturday','sunday','weekday']].dot(cal.columns[[1,2,5]] + '_').str.lower().apply(lambda x: x.strip('_'))

    ###find services that are weekday, weekend, or special service
    cal = cal.drop(columns=['saturday','sunday','weekday'])
    cal.dow = cal.dow.replace('', 'special')

    ###make service id a str
    cal.service_id = cal.service_id.astype(str)

    ###save service ids by dow for further exploration
    return cal 

def merge_gtfs_data(data,output_dir):
    """
    This function will merge the GTFS data into a single dataframe.
    
    Agency to agency_id -> Routes by agency_id -> trips by route_id -> stop_times by trip_id -> stops by stop_id
    """
    ##merge routes by agency id
    merged = pd.merge(data['agencies'], data['routes'], how='left', on='agency_id')
    print('Merged Agencies + Routes')
    ##merge trips by route id
    merged = pd.merge(merged, data['trips'], how='left', on='route_id')
    print('Merged Routes + Trips')
    ##merge stop_times by trip_id
    merged = pd.merge(merged, data['stop_times'], how='left', on='trip_id')
    print('Merged Agency + Route')
    ###merge stops by stop_id
    merged = pd.merge(merged, data['stops'], how='left', on='stop_id')
    print('Merged Agency + Route')
    ###merge dow variable onto data
    merged = pd.merge(merged, calendar_parse(output_dir)[['service_id','dow']], how='left', on='service_id')
    print('Added DOW variable, removed mon-sun designations in merged data.')
    ###drop data without a dow designation to simulate a 'normal' day
    prev = merged.shape[1]
    merged = merged.dropna(subset=['dow'])
    print('Dropped {} rows with null data.'.format(prev - merged.shape[1]))
    return merged

def filter_by_time(data, time=ALL_HOURS, dow=ALL_DOW):
    ###filter data by time to find the frequencies for the desired service period - dow and hours
    data['arrival_time'] = pd.to_timedelta(data['arrival_time'])
    filtered_data = data.loc[(data.arrival_time.between(pd.Timedelta(hours=time[0]), pd.Timedelta(hours=time[1]))) & (data['dow'].isin(dow))]
    return filtered_data

In [8]:
"""funcs for calculating frequencies and first/last arrivals"""
def first_last(data):
   trips = data.groupby(['trip_id', 'route_id']).departure_time.aggregate(['min', 'max', 'count'])
   trips = trips.merge(data[['trip_id', 'dow']], how='left', on='trip_id')
   print(trips.reset_index())

def calc_freq(data):
   """
   Script to take time-filtered data and calculate frequencies - returns dataframe with route id, service id, and aggregated freq data
   Data needs to be time-filtered before running this function as otherwise it will return gaps for the whole service day - if so use median 
   """
   data = data.copy()
   data['departure_time'] = pd.to_timedelta(data['departure_time'])
   data = data.sort_values(['stop_id', 'departure_time'])
   ###grouping so everything is held constant besides the trip_id - used for calculating previous departure and frequency 
   data['prev_depart'] = data.groupby(['route_id', 'service_id', 'direction_id', 'stop_id']).departure_time.shift(1)
   data['gap'] = data.departure_time - data.prev_depart
   ###return the list of routes with frequency information by service_id 
   freq = data.groupby(['route_short_name', 'route_id', 'service_id', 'dow']).gap.agg(['mean','median','min','max']).reset_index()
   for col in freq.select_dtypes('timedelta'):
      print(f'Rounding {col} to nearest minute...')
      freq[col] = freq[col].dt.round('min')
   return freq

def filter_by_freq(data, freq_short, freq_long):
   """
   Filters the freq dataframe produced in the calc_freq function by the desired frequency range in minutes
   """
   data = data.copy()
   freq_short, freq_long = pd.Timedelta(minutes=freq_short), pd.Timedelta(minutes=freq_long)
   data = data.loc[(data['median'] >= freq_short) & (data['median'] <= freq_long)]
   return data

In [20]:
import pandas as pd
gtfs_url = r'https://gtfs.sound.obaweb.org/prod/gtfs_puget_sound_consolidated.zip'
output_dir = '../data/pgc_gtfs'

merged_data = merge_gtfs_data(build_feed(), output_dir)

merged_data = filter_by_time(merged_data, dow=SATURDAY)



Merged Agencies + Routes
Merged Routes + Trips
Merged Agency + Route
Merged Agency + Route
Added DOW variable, removed mon-sun designations in merged data.
Dropped 0 rows with null data.


In [21]:
freqs = calc_freq(merged_data)

Rounding mean to nearest minute...
Rounding median to nearest minute...
Rounding min to nearest minute...
Rounding max to nearest minute...
